# Evaluation

Evaluates retrieval and answer quality for HF Model Finder:

- **Ground truth**: synthetic ML-task queries generated from 100 sampled models (Experiment A)
- **Experiment B**: keyword vs vector vs hybrid (RRF) retrieval, hit rate / MRR
- **Experiment E**: what text to embed for each model (raw card vs `embed_text` vs metadata only)
- **Experiment C**: two RAG prompt variants compared with an LLM judge
- **Experiment D**: LLM query rewriting before retrieval

Results are summarized in `README.md`.

Re-running this notebook top to bottom reproduces the retrieval numbers exactly: ground truth is only generated when `data/ground_truth.csv` doesn't exist (set `REGENERATE_GROUND_TRUTH = True` to force it). The LLM-judge and query-rewrite numbers involve fresh LLM calls, so they vary slightly between runs.

# Initialize

In [1]:
import json
import os
import random

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv()
openai_client = OpenAI()

In [2]:
with open("data/models.jsonl") as f:
    documents = [json.loads(line) for line in f]

len(documents)

3613

# Keyword index

In [3]:
from minsearch import Index

for doc in documents:
    doc["id_text"] = doc["id"].replace("/", " ").replace("-", " ").replace("_", " ").replace(".", " ")
    doc["task_text"] = doc["pipeline_tag"].replace("-", " ")
    doc["tags_text"] = " ".join(doc["topic_tags"])
    doc["languages_text"] = " ".join(doc["languages"])

index = Index(text_fields=["id_text", "task_text", "tags_text", "languages_text", "card_text"])
index.fit(documents)

# Vector index

Each model is embedded from its `embed_text` (built in `ingest.py`): task, library, languages, tags, then the first prose paragraph of the card. The embedder truncates at 128 tokens, so embedding the raw card would mostly embed its title and badges; Experiment E below checks this choice.

In [4]:
from minsearch import VectorSearch
from embedder import Embedder

embed = Embedder()

def embed_texts(texts, batch_size=50):
    X = []
    for i in tqdm(range(0, len(texts), batch_size)):
        X.extend(embed.encode_batch(texts[i:i + batch_size]))
    return np.array(X)

from embedding_cache import load_or_build

# Same content-keyed cache as app.py: rebuilt whenever any embed_text changes.
X = load_or_build([doc["embed_text"] for doc in documents], embed, "data/model_embeddings.npy")

vindex = VectorSearch()
vindex.fit(X, documents)

# Hybrid search

In [5]:
def rrf(search_results, k=60, num_results=10):
    scores = {}
    doc_map = {}

    for results in search_results:
        for rank, doc in enumerate(results):
            key = doc["id"]
            if key not in scores:
                scores[key] = 0
                doc_map[key] = doc
            scores[key] += 1 / (k + rank + 1)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in ranked[:num_results]]


def hybrid_search(query, num_results=10, k=60):
    # Fuse at least the top 10 of each list, even when fewer are returned.
    pool = max(num_results, 10)
    keyword_results = index.search(query, num_results=pool)
    vector_results = vindex.search(embed.encode(query), num_results=pool)
    return rrf([keyword_results, vector_results], k=k, num_results=num_results)

# Experiment A: ground truth

For 100 randomly sampled models, an LLM writes 3 search queries each that someone who needs *that* model might type.

**Several right answers.** Unlike a half-remembered title, an ML task often has several good models ("English sentiment classifier"). To keep the exact-id metric meaningful, the queries combine the task with the model's distinguishing traits (domain, language, a size or deployment constraint when the card states one). A secondary **task hit rate** also counts a result as relevant when its `pipeline_tag` matches the ground-truth model's.

**No leakage.** Queries may not mention the model id, org, model family/architecture names (Qwen, Llama, BERT, Whisper, CLIP...) or dataset names; otherwise retrieval becomes a trivial name lookup.

In [6]:
from pydantic import BaseModel
from evaluation_utils import llm_structured, calc_total_price, map_progress

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a developer looking for a pretrained model on the Hugging Face Hub.
Based on the model record below, write 3 short, natural search queries this
person might type if this model is exactly what they need.

Each query should describe the task they want to solve, combined with one or
two of this model's distinguishing traits: its domain, its language(s), or a
size / speed / deployment constraint when the record states one. Vary which
traits each query focuses on.

Do not mention the model id, the organization or author, model family or
architecture names (e.g. Qwen, Llama, Mistral, BERT, RoBERTa, Whisper, CLIP,
Stable Diffusion, YOLO), or dataset names. Use plain language or generic ML
vocabulary, the way someone would who doesn't know which model to use yet.

Keep each query under 15 words, casual like a real search box entry.
""".strip()

GEN_FIELDS = ["id", "pipeline_tag", "library_name", "languages", "license", "params", "topic_tags", "card_text"]

def generate_ground_truth(doc):
    user_prompt = json.dumps({k: doc[k] for k in GEN_FIELDS})
    out, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
    records = [{"question": q, "document": doc["id"]} for q in out.questions]
    return records, usage

In [7]:
from concurrent.futures import ThreadPoolExecutor

GROUND_TRUTH_PATH = "data/ground_truth.csv"
REGENERATE_GROUND_TRUTH = False

if REGENERATE_GROUND_TRUTH or not os.path.exists(GROUND_TRUTH_PATH):
    random.seed(0)
    sample = random.sample(documents, 100)

    with ThreadPoolExecutor(max_workers=6) as pool:
        results = map_progress(pool, sample, generate_ground_truth)

    rows, usages = [], []
    for records, usage in results:
        rows.extend(records)
        usages.append(usage)

    pd.DataFrame(rows).to_csv(GROUND_TRUTH_PATH, index=False)
    print(f"generated {len(rows)} queries, cost ${calc_total_price(usages):.4f}")

df_ground_truth = pd.read_csv(GROUND_TRUTH_PATH)
doc_by_id = {doc["id"]: doc for doc in documents}

# A re-ingest can drop models the ground truth was generated from (the corpus
# follows 30-day downloads). Skip those queries instead of crashing, and say how many.
missing = ~df_ground_truth["document"].isin(doc_by_id)
if missing.any():
    print(f"skipping {missing.sum()} queries whose model is no longer in data/models.jsonl")
df_ground_truth = df_ground_truth[~missing]
ground_truth = df_ground_truth.to_dict(orient="records")

print(len(ground_truth), "queries")
df_ground_truth.sample(10, random_state=0)

300 queries


,question,document
208,automatic answer evaluation for generated text,lucadiliello/BLEURT-20-D12
188,open source english assistant for transformers...,MaziyarPanahi/Llama-3-8B-Instruct-v0.8
12,fast image classification model for 224x224 im...,timm/resnet50.a1_in1k
221,Fast toxic message classification for English ...,JungleLee/bert-toxic-comment-classification
239,fast text toxicity detection for harmful conte...,unitary/toxic-bert
136,English biomedical NER for drug discovery,OpenMed/OpenMed-NER-ChemicalDetect-MultiMed-568M
230,Bengali automatic speech recognition with good...,arijitx/wav2vec2-xls-r-300m-bengali
206,fast masked text prediction for long documents,uw-madison/mra-base-512-4
52,lightweight chatbot for reasoning and code,microsoft/Phi-3-mini-4k-instruct
108,multilingual long document summarization for b...,ibm-granite/granite-4.0-tiny-preview


**Experiment B verdict: a trade-off, not a clear winner.**

- **Keyword search is competitive** (hit rate 0.270, close to vector's 0.297). Model queries share exact vocabulary with model cards ("NER", "Bengali", "toxic").
- **Hybrid finds the exact model most often** (0.337). The k sweep shows this isn't a tuning artifact.
- **Vector ranks a model of the right task first far more often** (task@1 0.763 vs hybrid 0.583), because keyword matches pull wrong-task models into the top ranks.

Exact-id metrics understate quality when several models are valid, so Experiment C settles this end to end.

# Experiment B: retrieval methods

In [8]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    task = doc_by_id[doc_id]["pipeline_tag"]
    results = search_function(q["question"])
    exact = [int(d["id"] == doc_id) for d in results]
    same_task = [int(d["pipeline_tag"] == task) for d in results]
    return exact, same_task

def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    pairs = [compute_relevance(q, search_function) for q in tqdm(ground_truth)]
    exact = [p[0] for p in pairs]
    task = [p[1] for p in pairs]
    return {
        "hit_rate": round(hit_rate(exact), 3),
        "mrr": round(mrr(exact), 3),
        "task_hit_rate": round(hit_rate(task), 3),
        "task_at_1": round(sum(t[0] for t in task) / len(task), 3),
    }

def text_search(query):
    return index.search(query, num_results=5)

def vector_search(query):
    return vindex.search(embed.encode(query), num_results=5)

def hybrid_search_fn(query):
    return hybrid_search(query, num_results=5)

In [9]:
results_b = {
    "keyword": evaluate(ground_truth, text_search),
    "vector": evaluate(ground_truth, vector_search),
    "hybrid (RRF k=60)": evaluate(ground_truth, hybrid_search_fn),
}
pd.DataFrame(results_b).T

,hit_rate,mrr,task_hit_rate,task_at_1
keyword,0.270,0.178,0.663,0.490
vector,0.297,0.195,0.897,0.763
hybrid (RRF k=60),0.337,0.221,0.863,0.583


RRF's `k` controls how much top ranks dominate. Sweeping it rules out a tuning artifact:

In [10]:
cache = [(q, index.search(q["question"], num_results=10), vindex.search(embed.encode(q["question"]), num_results=10))
         for q in tqdm(ground_truth)]

rows = []
for k in [1, 5, 10, 20, 60]:
    exact = [[int(d["id"] == q["document"]) for d in rrf([kw, vec], k=k, num_results=5)] for q, kw, vec in cache]
    rows.append({"k": k, "hit_rate": round(hit_rate(exact), 3), "mrr": round(mrr(exact), 3)})
pd.DataFrame(rows)

,k,hit_rate,mrr
0,1,0.340,0.222
1,5,0.340,0.221
2,10,0.337,0.221
3,20,0.337,0.221
4,60,0.337,0.221


# Experiment E: what to embed

Same embedder, same ground truth, vector search only. Five texts per model:

1. **raw card**: the cleaned card as-is (the embedder sees roughly its first 128 tokens)
2. **embed_text (shipped)**: a plain-language task description, library, languages, tags, then the first prose paragraph
3. **embed_text, bare task tag**: the same, but the task is written only as its tag name ("automatic speech recognition"), as in the first version of `ingest.py`
4. **metadata only**: task description, library, languages, tags; no card prose
5. **name + embed_text**: the repo name's words prefixed to the shipped text

In [11]:
import re
from ingest import TASK_DESCRIPTIONS

def task_label(tag):
    return TASK_DESCRIPTIONS.get(tag, tag.replace("-", " "))

def bare_tag_text(doc):
    return doc["embed_text"].replace(task_label(doc["pipeline_tag"]), doc["pipeline_tag"].replace("-", " "), 1)

def metadata_text(doc):
    parts = [task_label(doc["pipeline_tag"])]
    if doc["library_name"]:
        parts.append(doc["library_name"])
    if doc["languages"]:
        parts.append("Languages: " + (", ".join(doc["languages"]) if len(doc["languages"]) <= 8 else "multilingual"))
    if doc["topic_tags"]:
        parts.append("Tags: " + ", ".join(doc["topic_tags"][:8]))
    return ". ".join(parts)

def name_text(doc):
    return re.sub(r"[-_./]+", " ", doc["id"].split("/")[-1])

def vector_eval(X_variant):
    vi = VectorSearch()
    vi.fit(X_variant, documents)
    return evaluate(ground_truth, lambda query: vi.search(embed.encode(query), num_results=5))

results_e = {
    "raw card": vector_eval(embed_texts([doc["card_text"] for doc in documents])),
    "embed_text (shipped)": vector_eval(X),
    "embed_text, bare task tag": vector_eval(embed_texts([bare_tag_text(doc) for doc in documents])),
    "metadata only": vector_eval(embed_texts([metadata_text(doc) for doc in documents])),
    "name + embed_text": vector_eval(embed_texts([name_text(doc) + ". " + doc["embed_text"] for doc in documents])),
}
pd.DataFrame(results_e).T

,hit_rate,mrr,task_hit_rate,task_at_1
raw card,0.287,0.184,0.887,0.683
embed_text (shipped),0.297,0.195,0.897,0.763
"embed_text, bare task tag",0.270,0.171,0.883,0.740
metadata only,0.180,0.115,0.770,0.647
name + embed_text,0.310,0.208,0.883,0.770


**Experiment E verdict: ship `embed_text` with plain-language task descriptions.**

- **Task descriptions beat bare tag names on every metric** (hit rate 0.270 → 0.297, task@1 0.740 → 0.763). Bare tags embed almost identically when they share words: before this change, "transcribe English speech to text" retrieved only text-to-speech models. Spelling out each task's input → output direction ("transcribes spoken audio into written text") fixed that query and the aggregate numbers.
- **Card prose matters.** Metadata alone is clearly worst.
- **The raw card is weaker than `embed_text`**, especially at putting the right task first (task@1 0.683 vs 0.763). Its first 128 tokens are often a title, links and badges.
- **Name words are within noise.** Prefixing them gains 4 queries of exact hit rate out of 300, and task hit rate drops slightly. It's not worth the extra complexity.

# Experiment C: prompt variants and retrieval, judged end to end

Experiment B left a real trade-off: hybrid finds the exact model more often, while vector ranks the right task first far more often. An LLM judge settles it on what the user actually sees, the final recommendation. The judge also accepts a different model when it satisfies every constraint in the query as well as the ground-truth model does (see `judge.py`).

Three configurations, same 50 sampled queries:

- **v1 + vector**: open-ended prompt that allows several recommendations
- **v2 + vector**: forced single pick with a parseable `ANSWER:` line (the `rag_helper.py` default)
- **v2 + hybrid**: same prompt, hybrid (RRF) retrieval

In [12]:
class HybridIndexAdapter:
    def search(self, query, num_results=5):
        return hybrid_search(query, num_results=num_results)

In [13]:
from rag_helper import RAGBase, INSTRUCTIONS as INSTRUCTIONS_V2
from search_backends import VectorIndexAdapter
from judge import evaluate_answer
from evaluation_utils import calc_total_price

INSTRUCTIONS_V1 = '''
Your task is to help a user find a pretrained model on the Hugging Face
Hub based on a description of the ML task they want to solve.

Use the provided candidate models (retrieved by searching model cards,
tasks, and tags) to answer. Recommend the best-matching model(s) from the
candidates and briefly explain why they match, grounded only in the
retrieved information. If none of the candidates are a good match, say so
honestly instead of making one up.
'''.strip()

vector_index = VectorIndexAdapter(vindex, embed)
hybrid_index = HybridIndexAdapter()

configs = {
    "v1 + vector": RAGBase(index=vector_index, llm_client=openai_client, instructions=INSTRUCTIONS_V1),
    "v2 + vector": RAGBase(index=vector_index, llm_client=openai_client, instructions=INSTRUCTIONS_V2),
    "v2 + hybrid": RAGBase(index=hybrid_index, llm_client=openai_client, instructions=INSTRUCTIONS_V2),
}

def describe_model(doc):
    langs = ", ".join(doc["languages"][:8]) or "not specified"
    return f"{doc['id']} (task: {doc['pipeline_tag']}; languages: {langs})\n{doc['embed_text']}"

random.seed(2)
eval_sample = random.sample(ground_truth, 50)

In [14]:
def rag_with_usage(rag, query):
    results = rag.search(query)
    prompt = rag.build_prompt(query, results)
    response = openai_client.responses.create(model=rag.model, input=[
        {"role": "developer", "content": rag.instructions},
        {"role": "user", "content": prompt},
    ])
    return response.output_text, response.usage

def judge_one(rag, q):
    answer, rag_usage = rag_with_usage(rag, q["question"])
    verdict, _ = evaluate_answer(openai_client, q["question"], describe_model(doc_by_id[q["document"]]), answer)
    return {"question": q["question"], "document": q["document"], "answer": answer,
            "score": verdict.score, "reasoning": verdict.reasoning, "usage": rag_usage}

results_c = {}
for name, rag in configs.items():
    with ThreadPoolExecutor(max_workers=6) as pool:
        results_c[name] = map_progress(pool, eval_sample, lambda q: judge_one(rag, q))

summary_c = pd.DataFrame({
    name: {
        "good": sum(r["score"] == "good" for r in rows),
        "good_rate": round(sum(r["score"] == "good" for r in rows) / len(rows), 3),
        "rag_cost_usd": round(calc_total_price([r["usage"] for r in rows]), 4),
    }
    for name, rows in results_c.items()
}).T
summary_c

,good,good_rate,rag_cost_usd
v1 + vector,31.0,0.62,0.1617
v2 + vector,34.0,0.68,0.1174
v2 + hybrid,34.0,0.68,0.1117


**Experiment C verdict: ship the forced single pick (v2) with vector search.**

- **Hybrid adds nothing end to end.** v2 + vector and v2 + hybrid tie exactly (34/50), despite hybrid's exact-id edge in B, so the simpler vector search ships (no keyword index in the app).
- **v2 vs v1:** v2 scores 3 queries more than the open-ended v1, which is within noise at n=50. It is also ~27% cheaper per answer, and its `ANSWER:` line feeds the dashboard's "most recommended model" chart.

The judge's leniency rule shows up in the examples below: at least 4 of v2 + vector's 34 'good' verdicts are equally valid alternatives, not the ground-truth model. That confirms the exact-id hit rate understates real quality.

A few judged answers where the judge accepted a model other than the ground truth, to sanity-check the judge's leniency rule:

In [15]:
shown = 0
for r in results_c["v2 + vector"]:
    pick = re.search(r"ANSWER:\s*(.+)", r["answer"])
    pick = pick.group(1).strip() if pick else None
    if r["score"] == "good" and pick != r["document"] and shown < 4:
        shown += 1
        print("QUERY:", r["question"])
        print("GROUND TRUTH:", r["document"], "| PICKED:", pick)
        print("JUDGE:", r["reasoning"][:400])
        print()

QUERY: 7b conversational model with system prompt control
GROUND TRUTH: kaist-ai/janus-orpo-7b | PICKED: UCLA-AGI/Mistral7B-PairRM-SPPO-Iter3
JUDGE: The query asks for a 7B conversational model with system prompt control. The AI recommended `UCLA-AGI/Mistral7B-PairRM-SPPO-Iter3`, which is a 7.2B conversational English model and explicitly geared toward instruction/system-prompt following, so it satisfies the size and task constraints. Although it is not the ground-truth `kaist-ai/janus-orpo-7b`, it appears to be a valid alternative that matche

QUERY: general logic chatbot with stronger function calling
GROUND TRUTH: deepseek-ai/DeepSeek-R1-0528 | PICKED: baidu/ERNIE-4.5-21B-A3B-Thinking
JUDGE: The user wants a general logic chatbot with stronger function calling. The recommended model, baidu/ERNIE-4.5-21B-A3B-Thinking, is described as having improved logical reasoning and explicit function call support, which directly fits the added function-calling requirement. Although it is not the

# Experiment D: query rewriting

On this project's original corpus (plot synopses), rewriting made retrieval worse: short casual queries turned into long generic synopsis-style paragraphs that diluted the embedding. Model cards use fixed ML vocabulary, so a rewrite that maps casual phrasing onto task names ("spot names in text" → "token classification / named entity recognition") could plausibly help here. `query_rewrite.py` asks for a short phrase (under 30 words) in model-card terminology, not a paragraph.

Rewrites are cached so vector and hybrid search see identical rewritten queries.

In [16]:
from query_rewrite import rewrite_query

with ThreadPoolExecutor(max_workers=6) as pool:
    rewrites = dict(zip(
        [q["question"] for q in ground_truth],
        map_progress(pool, [q["question"] for q in ground_truth], lambda text: rewrite_query(openai_client, text)),
    ))

for q in ground_truth[:5]:
    print("RAW:      ", q["question"])
    print("REWRITTEN:", rewrites[q["question"]])
    print()

RAW:       multilingual named entity recognition for custom labels
REWRITTEN: multilingual token classification / named entity recognition custom labels

RAW:       lightweight entity extraction model for many languages
REWRITTEN: multilingual token classification / named entity recognition lightweight model

RAW:       token classification for open-ended NER in multilingual text
REWRITTEN: token classification / named entity recognition for open-ended multilingual text

RAW:       multilingual image captioning model for Japanese and Korean
REWRITTEN: multilingual image captioning Japanese Korean

RAW:       small vision language model for object detection and bounding boxes
REWRITTEN: vision-language object detection with bounding box prediction



In [17]:
results_d = {
    "vector": results_b["vector"],
    "vector + rewrite": evaluate(ground_truth, lambda query: vector_search(rewrites[query])),
    "hybrid": evaluate(ground_truth, hybrid_search_fn),
    "hybrid + rewrite": evaluate(ground_truth, lambda query: hybrid_search(rewrites[query], num_results=5)),
}
pd.DataFrame(results_d).T

,hit_rate,mrr,task_hit_rate,task_at_1
vector,0.297,0.195,0.897,0.763
vector + rewrite,0.270,0.175,0.867,0.770
hybrid,0.337,0.221,0.863,0.583
hybrid + rewrite,0.300,0.204,0.857,0.677


**Experiment D verdict: evaluated, not shipped. Rewriting slightly hurts.**

- **The rewrites map casual phrasing onto card vocabulary as intended** (see the examples above).
- **Retrieval still gets worse:** exact hit rate drops 0.297 → 0.270 for vector and 0.337 → 0.300 for hybrid, and task@1 barely moves.
- **The task descriptions already do the rewrite's job.** An earlier run of this notebook, before `embed_text` carried them, showed rewriting improving task@1 (0.743 → 0.780). Putting the translation from everyday words to ML task names on the index side is free at query time; the rewrite costs an extra LLM round trip.
- `RewritingVectorIndexAdapter` in `search_backends.py` remains available.